# Metrics, ROC-AUC, and overfitting checks

After `python -m src.pipeline.run_train`, open `artifacts/metrics.json`.

- **ROC-AUC** (often read as a 0–1 score): how well the model ranks fake vs reliable in the hold-out set. It is **not** “accuracy,” but many teams track both.
- **Precision / recall** (teacher mode in the UI): trade-offs when you change the decision threshold.
- **Overfitting signal:** train metrics much better than validation (especially AUC) → model may be memorizing quirks of the training split.

In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path("..").resolve()
metrics_path = PROJECT_ROOT / "artifacts" / "metrics.json"
m = json.loads(metrics_path.read_text(encoding="utf-8"))
m["overfitting_note"]

In [ ]:
def show(split):
    row = m["classical"][split]
    return {k: row[k] for k in ("precision", "recall", "f1", "roc_auc") if k in row}

show("train"), show("val"), show("test")

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is not None:
    rows = []
    for split in ("train", "val", "test"):
        r = m["classical"][split].copy()
        r["split"] = split
        rows.append(r)
    df = pd.DataFrame(rows).set_index("split")
    display(df[["precision", "recall", "f1", "roc_auc"]])
else:
    print("Install pandas to tabulate metrics.")

## Optional: plot train vs val AUC

If you log many runs with **MLflow**, compare runs in the MLflow UI instead of hand-plotting.

```bash
mlflow ui --backend-store-uri ./mlruns
```